### **Import Libraries**
This loads all the Python tools needed for **data handling and machine learning.**
##### **What problem are we solving?**
Python does not know how to work with data or build machine learning models by default.  
We must import the libraries that provide these capabilities.
##### **What happens step by step**
1. **`pandas`** is imported to work with tables of data  
2. **`numpy`** is imported to handle missing values and numerical operations  
3. Tools from **scikit-learn (`sklearn`)** are imported to:
   - split data into training and validation sets  
   - clean and prepare different types of data  
   - build a machine learning pipeline  
   - train a Logistic Regression model  
   - evaluate model performance using F1 score 
##### **Why it matters**
- Without these libraries, we cannot load, clean, or model data  
- Each library has a specific role in the machine learning process  
- Importing everything at the start makes the notebook organised and readable
This step ensures:
- the right tools are available  
- later code runs without errors  
- the workflow is clear and reproducible
### **In very simple words**
> “We load the tools we need before we start working with data.”


In [ ]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression

ModuleNotFoundError: No module named 'sklearn'

### Load the Datasets
This loads the data files needed to train and test the machine learning model.
#### What problem are we solving?
Before a model can learn or make predictions, the data must be read into Python.  
The data is stored in CSV files, which need to be loaded as tables.
1. The file `train.csv` is read into Python  
2. The file `test.csv` is read into Python  
3. The datasets are stored as tables called `train` and `test`  
4. The shape of each dataset is printed to show how many rows and columns they contain  
5. The first few rows of the training data are displayed  
#### Why we inspect the data
- Confirms the files loaded correctly  
- Helps us understand the structure of the data  
- Allows us to spot obvious issues early 
#### Why it matters
- The model **learns patterns** from the `train` dataset  
- The model **makes predictions** on the `test` dataset  
- Checking the data early prevents mistakes later  
This step ensures:
- the data is loaded correctly  
- the structure is understood  
- the workflow starts on a solid foundation 
### In very simple words
> “We load the data into Python and quickly check it before using it.”


In [6]:
# Load data
train = pd.read_csv("../data/train.csv")
test  = pd.read_csv("../data/test.csv")

print(train.shape, test.shape)
# Shows the first few rows
# Always inspect data before using it.
train.head()

(2000, 12) (800, 11)


,id,dob_given,photo_added,can_send_message,gender_added,name_added,profile_private,interests_mentioned,mobile_given,friends_count,posts_count,outcome
0,1,1,1,0,1,1,1,0,1,1321,416,0
1,2,Unknown,0,0,1,0,1,1,1,1286,760,1
2,3,Unknown,0,1,0,0,1,1,1,7430,622,1
3,4,0,0,1,0,1,0,0,0,2068,493,0
4,5,0,0,0,0,1,1,0,1,417,673,1


In [7]:
# Shows the first few rows
# Always inspect data before using it.
test.head()

,id,dob_given,photo_added,can_send_message,gender_added,name_added,profile_private,interests_mentioned,mobile_given,friends_count,posts_count
0,1,0,0,1,1,0,1,1,1,757,471
1,2,0,0,1,1,1,1,1,1,898,350
2,3,0,1,1,1,1,1,0,Unknown,465,1662
3,4,0,1,1,1,1,0,0,1,3935,2830
4,5,0,0,1,1,0,1,1,0,1379,450


### Replace "Unknown" with Missing Values
This converts the text value **`"Unknown"` into a proper missing value that Python can understand.
Some columns contain the word **`"Unknown"` instead of real information.  
Although humans understand what `"Unknown"` means, **machine learning models do not**.
To the model, `"Unknown"` is just another word, not a missing value.
#### What happens step by step
1. The code searches the dataset for the text `"Unknown"`  
2. Every `"Unknown"` value is replaced with `NaN`  
3. `NaN` is Python’s standard way of representing missing data  
4. This replacement is applied to both the training and test datasets 
#### What is `NaN`?
- `NaN` means **“Not a Number”**  
- It tells Python that the value is missing  
- Machine learning tools know how to handle `NaN` properly 
#### Why it matters
- Machine learning models cannot understand words like `"Unknown"`  
- Missing values must be clearly marked before processing  
- This allows missing data to be handled safely later (for example, by filling or ignoring it) 
This step helps the model:
- correctly recognise missing information  
- avoid treating `"Unknown"` as a real category  
- prepare the data for proper cleaning and training 
#### In very simple words
> “We replace the word ‘Unknown’ with a special marker so the model knows the value is missing.”

In [8]:
train.replace("Unknown", np.nan, inplace=True)
test.replace("Unknown", np.nan, inplace=True)

#### Convert Counts to Numbers
This makes sure that **`friends_count`** and **`posts_count`** are stored as numbers.

Some values in the **`friends_count`** and **`posts_count`** columns may not be valid numbers.  
They might be text, symbols, or incorrectly formatted values.
Machine learning models **cannot perform calculations on text**, so these values must be fixed.
#### What happens step by step

1. The **`friends_count`** column is converted into numeric values  
2. Any value that cannot be converted becomes **`NaN`** (missing value)  
3. The same process is applied to the **`posts_count`** column  
4. This conversion is done for both the training and test datasets 
#### Why bad values become **`NaN`**
- **`NaN`** clearly marks values that are invalid or missing  
- These values can be safely handled later during preprocessing  
- This avoids crashes or incorrect calculations
#### Why it matters
- Mathematical models only work with numbers  
- Invalid values can break training or reduce accuracy  
- Converting values early makes the data clean and reliable 
This step helps the model:
- perform mathematical calculations correctly  
- handle missing data safely  
- learn meaningful patterns
#### **In very simple words**
> “We turn counts into numbers so the model can do maths, and mark bad values as missing.”


In [9]:
train["friends_count"] = pd.to_numeric(train["friends_count"], errors="coerce")
test["friends_count"]  = pd.to_numeric(test["friends_count"], errors="coerce")

train["posts_count"] = pd.to_numeric(train["posts_count"], errors="coerce")
test["posts_count"]  = pd.to_numeric(test["posts_count"], errors="coerce")


#### Limit very large and very small values
This reduces the effect of unusually large values in the **`friends_count`** column.
#### What problem are we solving?
Some users may have an extremely high number of friends.  
These values are called **outliers**.
Outliers can:
- dominate the learning process  
- pull the model in the wrong direction  
- reduce prediction accuracy 

1. The model looks at the **`friends_count`** values in the **training data**  
2. It finds the value below which **99% of users fall** (99th percentile)  
3. It finds the value above which only **1% of users fall** (1st percentile)  
4. Any value higher than the upper limit is reduced to that limit  
5. Any value lower than the lower limit is increased to that limit  
6. The same limits are applied to the test data to keep things consistent
##### Why we use percentiles
- Percentiles describe what is **normal** in the data  
- They help ignore extreme cases without deleting data  
- This keeps most real behaviour unchanged 
## Why it matters
- Machine learning models are sensitive to extreme values  
- Very large numbers can overpower smaller but important patterns  
- Clipping improves stability and fairness in learning 
This step helps the model:
- learn typical behaviour  
- avoid being misled by extreme cases  
- make more reliable predictions 
##### In very simple words
> “We stop unusually **large or small numbers** from confusing the model by keeping them within a normal range.”


In [10]:
train["friends_count"] = train["friends_count"].clip(
    train["friends_count"].quantile(0.01),
    train["friends_count"].quantile(0.99)
)

test["friends_count"] = test["friends_count"].clip(
    train["friends_count"].quantile(0.01),
    train["friends_count"].quantile(0.99)
)



### Separate features **(X) and target (y)**
1. X -> information used to predict
2. y -> the answer (outcome)
### Why it matters:
#### The model learns patterns from **X** in order to predict **y**.

In [11]:
X = train.drop(columns=["outcome", "id"])
y = train["outcome"]


### Select numeric and categorical columns
1. Identifies all **columns in X** that contain numeric values, such as counts like friends and posts
2. Identifies all **columns in X** that do not contain numbers, such as text fields or yes/no indicators
3. Separates the dataset into numeric data and categorical data so each can be processed correctly
#### Why it matters:
##### Numeric and categorical data behave differently, so they must be cleaned and prepared in different ways before training a machine learning model.

In [12]:
num_cols = X.select_dtypes(include="number").columns
cat_cols = X.select_dtypes(exclude="number").columns


#### **Prepare Data Using ColumnTransformer**
This cell tells the computer how to clean different types of data before training the model.
##### What happens step by step
- A **ColumnTransformer** is created to apply different rules to different columns  
- Numeric columns and text columns are handled separately 
#### Numeric columns **(`"num"`)**
- Columns listed in **`num_cols`** are selected  
- Missing numeric values are replaced with the **median (middle value)**  
- This ensures all numeric columns contain valid numbers  
#### Categorical columns **(`"cat"`)**
- Columns listed in **`cat_cols`** are selected  
- Missing text values are replaced with the **most common value**  
- Text values are converted into numbers using **One-Hot Encoding** 
#### **One-Hot Encoding**
- Each text value becomes a separate numeric column  
- This allows the model to understand text information 
#### **Handling unknown values**
- New categories during prediction are safely ignored  
- This prevents errors when predicting on new data 
#### Why it matters
- Machine learning models cannot work with missing values  
- Machine learning models cannot understand text  
- Numeric and categorical data must be treated differently

This step ensures the data is:
- **complete**
- **numeric**
- **safe for both training and prediction**


In [13]:
preprocess = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols)
])


#### Logistic Regression is a simple machine-learning model for yes/no decisions.
##### It answers questions like:
- Is this a bot or a human?
- Is this spam or not spam?
- Is this fraud or not fraud?
##### What it actually does
- Looks at input features (friends, posts, profile info)
- Combines them into one score
- Converts that score into a probability between 0 and 1
- If probability ≥ 0.5 → predicts 1
- Else → predicts 0

## Logistic Regression predicts the probability of a yes/no outcome.

#### **Build the Machine Learning Model Pipeline**
This creates the final machine learning model by combining data preparation and learning into one process.
##### **What happens step by step**
- A **Pipeline** is created to control the order of actions  
- The first step (`"prep"`) cleans and prepares the data using the preprocessing rules  
- The second step (`"clf"`) trains a Logistic Regression model on the prepared data 
##### **Preprocessing step (`"prep"`)**
- Uses the `preprocess` object created earlier  
- Handles missing values and converts text into numbers  
- Ensures the data is ready for the model  
##### **Model step (`"clf"`)**
- Uses **Logistic Regression** to learn from the data  
- `max_iter=5000` gives the model enough time to finish learning  
- Produces a binary prediction (0 or 1) 
##### **Why it matters**
- Ensures the same data preparation is used during training and prediction  
- Prevents mistakes caused by inconsistent data handling  
- Makes the model easier to use and reuse
This step creates a model that is:
- **consistent**
- **reliable**
- **ready for prediction**


In [14]:

model = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=5000))
])



#### **Split Data into Training and Validation Sets**
This cell divides the data into two parts: one for learning and one for testing.
- The dataset is split into **training data (80%)** and **validation data (20%)**  
- **`X_train`** and **`y_train`** are used to train the model  
- **`X_val`** and **`y_val`** are used to test the model 
#### Stratified split
- `stratify=y` keeps the same class balance in both sets  
- Each set contains a similar proportion of 0s and 1s
#### Important note
- This step will **fail** if `y` contains only one class meaning only 0's or 1's 
- Machine learning models need examples of **all classes** to learn 
#### Why it matters
- Tests the model on data it has not seen before  
- Prevents biased evaluation  
- Ensures fair and reliable performance results
This step ensures:
- **balanced data**
- **fair testing**
- **reliable evaluation**


In [15]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


### Check the **Sizes** of the **Split Data**
This cell checks the size of the training and validation datasets.
- Prints the number of rows and columns in **`X_train`** 
- Prints the number of rows and columns in **`X_val`** 
- Prints the number of values in **`y_train`**  
- Prints the number of values in **`y_val`**
#### Why it matters

- Confirms the data was split correctly  
- Ensures features **(`X`)** and targets **(`y`)** match in size  
- Helps detect mistakes early before training the model  

This step ensures:
- **correct data split**
- **matching inputs and outputs**
- **no hidden errors**


In [16]:
#Check sizes
print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)

X_train shape: (1600, 10)
X_val shape: (400, 10)
y_train shape: (1600,)
y_val shape: (400,)


#### Check Class Distribution
This checks how many examples of each class **(0 and 1)** exist in the training and validation data.
##### What happens step by step
- Counts how many times each outcome appears in **`y_train`**  
- Counts how many times each outcome appears in **`y_val`**  
- Displays the number of **0s and 1s** in each dataset
##### Why it matters
- Confirms that both classes are present in the data  
- Explains why a model may fail if only one class exists  
- Ensures the model has enough examples to learn from
This step ensures:
- **balanced classes**
- **successful model training**
- **reliable evaluation**


In [17]:
print("y_train values:")
print(y_train.value_counts())

print("\ny_val values:")
print(y_val.value_counts())

y_train values:
outcome
0    806
1    794
Name: count, dtype: int64

y_val values:
outcome
0    201
1    199
Name: count, dtype: int64


#### Preview Training Labels
This cell shows the first few values of the training target **(`y_train`).**
- Displays the first five values in **`y_train`**  
- Shows whether each example is labelled as **0 or 1** 
##### Why it matters
- Confirms that the target variable contains valid labels  
- Helps visually verify that both classes **(0 and 1)** are present  
- Provides reassurance before training the model 
This step ensures:
- **labels are correct**
- **data looks as expected**
- **no hidden issues before training**


In [18]:
y_train.head()

1268    0
994     0
1062    0
1258    1
266     1
Name: outcome, dtype: int64

#### Train the Model and Evaluate with F1 Score
This trains the machine learning model and checks how well it performs.
- The model is trained using **`X_train`** and **`y_train`**  
- The trained model makes predictions on **`X_val`** 
- The **F1 score** is calculated by comparing predictions with **true labels** 
##### Why it matters
- Confirms the model is learning patterns from the data  
- Measures how accurate the predictions are  
- Helps decide if the model performance is acceptable 
This step ensures:
- **successful training**
- **meaningful evaluation**
- **reliable performance results**


In [19]:
model.fit(X_train, y_train)
preds = model.predict(X_val)

print("F1 score:", f1_score(y_val, preds))


F1 score: 0.6683417085427136


#### Generate Predictions and Save Results
This uses the trained model to make predictions on new data and saves the results.
- Removes the **`id`** column from the test data before prediction  
- Uses the trained model to predict outcomes for the test data  
- Combines each prediction with its corresponding **`id`**
- Saves the results to a CSV file called **`submissions.csv`** 
- Displays the first few rows of the results
#### Why it matters
- Ensures the model only sees useful features when predicting  
- Keeps predictions linked to the correct records  
- Creates a file that can be shared or submitted
This step ensures:
- **correct predictions**
- **clear output**
- **ready-to-use results**


In [21]:
test_features = test.drop(columns=["id"])
test_preds = model.predict(test_features)


submissions = pd.DataFrame({
    "id": test["id"],
    "outcome": test_preds
})

submissions.to_csv("../outputs/submissions.csv", index=False)
submissions.head()


,id,outcome
0,1,0
1,2,0
2,3,1
3,4,0
4,5,1
